# ImprovedGS + C2F + pose-aware cho VAI public/private

Notebook này chọn một/nhiều scene, preprocess, train, render test pose ra cả JPEG và PNG, đánh giá public ground truth và đóng gói hai định dạng thành hai ZIP riêng.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

PHASE_DIR = Path('/kaggle/input/datasets/xuanph/phase1/phase1')
WORK_ROOT = Path('/kaggle/working')
REPO_DIR = WORK_ROOT / 'Improved-GS'
REPO_BRANCH = 'main'

In [ ]:
# ========== CHINH TOAN BO THAM SO CHAY TAI CELL NAY ==========
SET_NAME = 'public_set'  # Vi du: 'public_set', 'private_set1'
SCENE_NAMES = ['HCM0204']  # Mot scene, nhieu scene, hoac [] de chay tat ca scene trong set
EVALUATE = SET_NAME == 'public_set'
REQUIRE_GT = EVALUATE
SAVE_PNG = True

DATA_SET = PHASE_DIR / SET_NAME
if not DATA_SET.is_dir():
    raise FileNotFoundError(f'Khong tim thay VAI set: {DATA_SET}')
AVAILABLE_SCENES = sorted(
    path.name for path in DATA_SET.iterdir()
    if path.is_dir() and (path / 'test' / 'test_poses.csv').is_file()
)
if len(set(SCENE_NAMES)) != len(SCENE_NAMES):
    raise ValueError(f'SCENE_NAMES bi trung: {SCENE_NAMES}')
missing_scenes = sorted(set(SCENE_NAMES) - set(AVAILABLE_SCENES))
if missing_scenes:
    raise ValueError(f'Khong tim thay scene trong {SET_NAME}: {missing_scenes}')
SELECTED_SCENES = list(SCENE_NAMES) if SCENE_NAMES else AVAILABLE_SCENES
if not SELECTED_SCENES:
    raise ValueError(f'Khong co scene hop le trong {DATA_SET}')

DATA_ROOT = WORK_ROOT / 'vai_cleaned' / SET_NAME
MODEL_ROOT = WORK_ROOT / 'vai_models' / SET_NAME
RENDER_ROOT = WORK_ROOT / 'vai_renders' / SET_NAME
PNG_ROOT = WORK_ROOT / 'vai_png' / SET_NAME
EVAL_ROOT = WORK_ROOT / 'vai_eval' / SET_NAME
RUNTIME_CONFIG_PATH = WORK_ROOT / f'vai_{SET_NAME}.runtime.json'

VAI_CONFIG = {
    'data_root': str(DATA_ROOT),
    'output_root': str(MODEL_ROOT),
    'repeat': 1,
    'python_executable': sys.executable,
    'gpu_auto_select': False,
    'gpu_id': 0,
    'stop_on_error': True,
    'run_postprocess': True,
    'postprocess_script': 'vai_render.py',
    'select_best_repeat_by_psnr': False,
    'train_args': {
        'training_method': 'improvedgs',
        'iterations': 30000,
        'coarse_to_fine': True,
        'coarse_to_fine_middle_iter': 2000,
        'coarse_to_fine_full_iter': 5000,
        'pose_aware_sampling': True,
        'pose_aware_k': 3,
        'pose_aware_extra_fraction': 0.25,
        'pose_aware_max_repeat': 2,
        'pose_aware_angle_weight': 0.25,
        'eval': False,
        'resolution': -1,
        'data_device': 'cpu',
        'budget': 3000000,
        'progress_bar_width': 100,
        'report_lpips_test': False,
        'report_lpips_train': False,
    },
    'postprocess_args': {
        'output_root': str(RENDER_ROOT),
        'eval_root': str(EVAL_ROOT),
        'output_extension': 'csv',
        'save_png': SAVE_PNG,
        'png_root': str(PNG_ROOT),
        'redistort_interpolation': 'bicubic',
        'sharpen_amount': 1.0,
        'sharpen_sigma': 0.60,
        'jpeg_quality': 95,
        'jpeg_subsampling': 2,
        'evaluate': EVALUATE,
        'require_gt': REQUIRE_GT,
        'overwrite': True,
        'psnr_max': 40.0,
        'lpips_net': 'alex',
    },
    'scenes': [{'name': name} for name in SELECTED_SCENES],
}

RUNTIME_CONFIG_PATH.write_text(
    json.dumps(VAI_CONFIG, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(f'Runtime config: {RUNTIME_CONFIG_PATH}')
print(f'Available scenes in {SET_NAME}: {AVAILABLE_SCENES}')
print(f'Selected scenes: {SELECTED_SCENES}')
print(json.dumps(VAI_CONFIG, ensure_ascii=False, indent=2))

In [ ]:
# Clone code chinh cua su phu.
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--recursive', '--branch', REPO_BRANCH,
        'https://github.com/mdd206/Improved-GS.git', str(REPO_DIR)
    ], check=True)
os.chdir(REPO_DIR)
print(Path.cwd())

In [ ]:
# Cai COLMAP CLI. APT duoc de verbose de exit code 100 khong che mat nguyen nhan.
def install_colmap_with_apt():
    apt_env = os.environ.copy()
    apt_env['DEBIAN_FRONTEND'] = 'noninteractive'
    common_options = [
        '-o', 'Dpkg::Use-Pty=0',
        '-o', 'Dpkg::Lock::Timeout=120',
        '-o', 'Acquire::Retries=3',
    ]
    update_result = subprocess.run(
        ['apt-get', *common_options, 'update'],
        check=False, env=apt_env,
    )
    if update_result.returncode != 0:
        print(f'apt-get update that bai voi ma {update_result.returncode}; thu Conda fallback.')
        return False

    install_command = [
        'apt-get', *common_options, 'install', '-y',
        '--no-install-recommends', 'colmap',
    ]
    install_result = subprocess.run(install_command, check=False, env=apt_env)
    if install_result.returncode == 0:
        return True

    print(f'APT tra ma {install_result.returncode}; repair dependency roi thu lai mot lan.')
    subprocess.run(
        ['apt-get', *common_options, '--fix-broken', 'install', '-y'],
        check=False, env=apt_env,
    )
    retry_result = subprocess.run(install_command, check=False, env=apt_env)
    return retry_result.returncode == 0


def install_colmap_with_conda():
    # Prefix rieng tranh thay doi Python/PyTorch cua notebook.
    conda_cli = next(
        (candidate for candidate in ('micromamba', 'mamba', 'conda') if shutil.which(candidate)),
        None,
    )
    if conda_cli is None:
        print('Khong tim thay micromamba/mamba/conda de fallback.')
        return False

    colmap_env = WORK_ROOT / 'colmap-env'
    action = 'install' if (colmap_env / 'conda-meta').is_dir() else 'create'
    result = subprocess.run([
        conda_cli, action, '-y', '-p', str(colmap_env),
        '-c', 'conda-forge', 'colmap',
    ], check=False)
    colmap_bin = colmap_env / 'bin' / 'colmap'
    if result.returncode == 0 and colmap_bin.is_file():
        os.environ['PATH'] = str(colmap_bin.parent) + os.pathsep + os.environ.get('PATH', '')
        return True
    return False


if shutil.which('colmap') is None:
    install_colmap_with_apt()
if shutil.which('colmap') is None:
    install_colmap_with_conda()
if shutil.which('colmap') is None:
    raise RuntimeError('Khong the cai COLMAP CLI. Kiem tra log APT, bat Internet va dung luong /kaggle/working.')
print('COLMAP:', shutil.which('colmap'))

# Cai dependency Python va CUDA extension.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy==1.26.1', 'opencv-python==4.10.0.82',
                'setuptools==69.5.1', 'tqdm', 'plyfile'], check=True)
for package_dir in [
    'submodules/diff-gaussian-rasterization',
    'submodules/simple-knn',
    'submodules/fused-ssim',
]:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    package_dir, '--no-build-isolation'], check=True)

In [ ]:
# Chuyen SIMPLE_RADIAL thanh PINHOLE RGBA cho cac scene da chon.
subprocess.run([
    sys.executable, 'vai_preprocess.py',
    '--input', str(DATA_SET),
    '--output', str(DATA_ROOT),
    '--subset', *SELECTED_SCENES,
], check=True)

In [ ]:
# In command de kiem tra path va tham so truoc khi train.
subprocess.run([
    sys.executable, 'run.py',
    '-c', str(RUNTIME_CONFIG_PATH),
    '--dry_run',
], check=True)

In [ ]:
# Train va render tung scene; public co eval, private bo qua eval theo config.
env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
subprocess.run([
    sys.executable, 'run.py',
    '-c', str(RUNTIME_CONFIG_PATH),
], check=True, env=env)

In [ ]:
# Validate va dong goi JPEG dung ten/duoi trong test_poses.csv.
jpeg_zip = WORK_ROOT / f'{SET_NAME}_jpeg.zip'
subprocess.run([
    sys.executable, 'vai_package.py',
    '--phase_dir', str(PHASE_DIR),
    '--set_name', SET_NAME,
    '--submission_dir', VAI_CONFIG['postprocess_args']['output_root'],
    '--zip_path', str(jpeg_zip),
    '--subset', *SELECTED_SCENES,
    '--output_extension', VAI_CONFIG['postprocess_args']['output_extension'],
], check=True)
print(jpeg_zip)

# Validate va dong goi PNG lossless thanh ZIP rieng.
if SAVE_PNG:
    png_zip = WORK_ROOT / f'{SET_NAME}_png.zip'
    subprocess.run([
        sys.executable, 'vai_package.py',
        '--phase_dir', str(PHASE_DIR),
        '--set_name', SET_NAME,
        '--submission_dir', str(PNG_ROOT),
        '--zip_path', str(png_zip),
        '--subset', *SELECTED_SCENES,
        '--output_extension', 'png',
    ], check=True)
    print(png_zip)

# Public set co JSON metric; private set khong co ground truth nen bo qua.
if EVALUATE:
    missing_eval = [name for name in SELECTED_SCENES if not (EVAL_ROOT / f'{name}.json').is_file()]
    if missing_eval:
        raise FileNotFoundError(f'Thieu evaluation JSON cho scene: {missing_eval}')
    eval_zip = Path(shutil.make_archive(
        str(WORK_ROOT / f'{SET_NAME}_eval'),
        'zip',
        root_dir=EVAL_ROOT,
    ))
    print(eval_zip)